# Ball-and-Beam Pipeline (compact)

Uses the `bab` package. See `bab_dlc_to_pytorch.ipynb` for the full detailed version.

| Part | Description |
|------|-------------|
| A | DLC training + analysis + theta extraction |
| C | TF checkpoint → PyTorch conversion |
| D | PyTorch inference + theta plot |
| — | Comparison: DLC TF vs PyTorch |

In [ ]:
# ── Setup ──

from pathlib import Path
import matplotlib.pyplot as plt

from bab import (
    run_dlc_training,
    run_dlc_analysis,
    extract_theta_from_dlc,
    convert_dlc_tf_to_pytorch,
    analyze_video_pytorch,
)

working_dir = "/Users/eg75agon/Downloads/Project_helon"
video_path = str(Path(working_dir) / "swept_sine_ready.MOV")
config_path = str(
    Path(working_dir)
    / "bab_bar_2pts_dlc3-Dani_F-2026-02-24"
    / "config.yaml"
)
snapshot_path = str(
    Path(working_dir)
    / "bab_bar_2pts_dlc3-Dani_F-2026-02-24"
    / "dlc-models" / "iteration-0"
    / "bab_bar_2pts_dlc3Feb24-trainset95shuffle1"
    / "train" / "snapshot-500"
)

assert Path(video_path).exists(), f"Video not found: {video_path}"
assert Path(config_path).exists(), f"Config not found: {config_path}"
print("Ready.")

In [ ]:
# ── Part A: DLC Training + Analysis ──
# Skip this cell if DLC has already been trained (snapshot-500 exists)

run_dlc_training(config_path, video_path, maxiters=500)
run_dlc_analysis(config_path, video_path)

In [ ]:
# ── Part A: Extract theta from DLC results ──

dlc_tf_df = extract_theta_from_dlc(video_path, working_dir, fps=30.0)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dlc_tf_df["t_s"], dlc_tf_df["theta_deg"], linewidth=0.8)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Theta [degrees]")
ax.set_title("Beam angle – DLC TensorFlow ResNet-50")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Part C: Convert DLC TF checkpoint → PyTorch ──

save_pt_path = str(Path(working_dir) / "checkpoints" / "dlc_converted.pt")
dlc_model = convert_dlc_tf_to_pytorch(
    ckpt_path=snapshot_path,
    num_keypoints=2,
    save_path=save_pt_path,
)
print(f"Saved: {save_pt_path}")

In [ ]:
# ── Part D: PyTorch Inference ──

dlc_pt_csv = str(Path(working_dir) / "theta_dlc_pytorch.csv")
dlc_pt_df = analyze_video_pytorch(
    dlc_model,
    video_path=video_path,
    output_csv=dlc_pt_csv,
    fps=30.0,
    device="cpu",
    model_type="dlc",
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dlc_pt_df["t_s"], dlc_pt_df["theta_deg"], linewidth=0.8)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Theta [degrees]")
ax.set_title("Beam angle – Converted DLC PyTorch ResNet-50")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparison: DLC TF vs PyTorch ──

import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Overlay
axes[0].plot(dlc_tf_df["t_s"], dlc_tf_df["theta_deg"],
             linewidth=0.8, label="DLC TensorFlow", alpha=0.8)
axes[0].plot(dlc_pt_df["t_s"], dlc_pt_df["theta_deg"],
             linewidth=0.8, label="PyTorch (converted)", alpha=0.8, linestyle="--")
axes[0].set_ylabel("Theta [degrees]")
axes[0].set_title("DLC TF vs Converted PyTorch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Difference
n = min(len(dlc_tf_df), len(dlc_pt_df))
diff = dlc_tf_df["theta_deg"].values[:n] - dlc_pt_df["theta_deg"].values[:n]
axes[1].plot(dlc_tf_df["t_s"].values[:n], diff, linewidth=0.6, color="red")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Difference [degrees]")
axes[1].set_title(f"Difference (mean={np.nanmean(diff):.4f}, std={np.nanstd(diff):.4f})")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()